# Interactive run-log explorer

Use this notebook to load one or more monkey-game runs from a directory, a `.zip` download, or an individual `.json` trial. It validates the schema, separates counter continuity from timing stalls, checks logged input-to-movement response, compares runs, and provides selectable 2D and 3D views.

The interactive controls use [ipywidgets](https://ipywidgets.readthedocs.io/en/stable/examples/Using%20Interact.html); figures use [Plotly](https://plotly.com/python/dropdowns/) and its [3D scatter support](https://plotly.com/python/3d-scatter-plots/). Analysis logic lives in `tools/utils/run_analysis.py`, while `tools/utils/run_dashboard.py` only defines the notebook UI.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'controller_main.js').exists() and (candidate / 'tools').is_dir():
            return candidate
    raise RuntimeError('Could not find the repository root from the notebook working directory.')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tools.utils.run_analysis import (
    AnalysisConfig, filter_frames, load_runs, plot_frame_pacing, timing_events,
    plot_run_comparison, plot_trial_overview, plot_trajectory_3d,
)
from tools.utils.run_dashboard import RunBrowser, RunDashboard

pd.set_option('display.max_columns', 100)
print(f'Repository: {REPO_ROOT}')

Repository: /home/ggil/PhD/monkey_3d_game


## 1. Browse and load interactively

Run the next cell, select any combination of discovered sources, and click **Load selected**. You can change the folder to `~/Downloads` or paste one absolute ZIP/directory path per line in **Extra paths**. Compatibility errors are shown before the plots.

In [2]:
browser = RunBrowser(search_root=REPO_ROOT / 'out', repo_root=REPO_ROOT)
browser.display()

## 2. Reproducible/programmatic analysis

This cell loads the two supplied test runs when present. Replace `SOURCES` with any mix of ZIPs, directories, or JSON files. `dataset.frames`, `dataset.trials`, `dataset.levels`, `dataset.trial_summary`, `dataset.run_summary`, and `dataset.issues` are ordinary pandas DataFrames.

In [ ]:
SOURCES = [
    REPO_ROOT / 'out' / 'test_final_virgilo.zip',
    REPO_ROOT / 'out' / 'test_from_phone_virgilio.zip',
]
SOURCES = [path for path in SOURCES if path.exists()]
if not SOURCES:
    raise FileNotFoundError('Set SOURCES to at least one run directory, ZIP, or JSON file.')

config = AnalysisConfig(expected_hz=60, late_multiplier=1.5, severe_multiplier=3.0)
dataset = load_runs(SOURCES, config=config)
display(dataset.run_summary.round(4))
print(f'{dataset.frames.run_id.nunique()} runs | {dataset.frames.trial_uid.nunique()} trials | {len(dataset.frames):,} frames')

### Diagnostics and compatibility

Errors identify incompatible/missing required fields; warnings identify questionable timing or continuity; info rows document limitations that matter when interpreting the log.

In [ ]:
display(dataset.issues)
display(timing_events(dataset, active_only=True, limit=20).round(4))
display(
    dataset.trial_summary
    .sort_values('dt_max_ms', ascending=False)
    [['run_id', 'level_index', 'trial_run_counter', 'n_frames', 'late_frames',
      'dt_p99_ms', 'dt_max_ms', 'dt_lag1_autocorr', 'frame_gaps', 'render_gaps']]
    .head(20)
    .round(4)
)

## 3. Interactive views

The same dashboard used by the browser can be created from a programmatically loaded dataset. In **Flexible 2D** and **3D**, choose fields for the axes/color and click the update button. Plotly provides hover details, legend filtering, zoom, pan, rotation, and image export.

In [ ]:
dashboard = RunDashboard(dataset)
dashboard.display()

## 4. Direct plot examples

These calls are useful when you want a reproducible figure without widget state. The 3D example deliberately uses state/timing fields because the physical camera position is constant during object rotation in these runs.

In [ ]:
display(plot_frame_pacing(dataset))
display(plot_run_comparison(dataset, metric='dt_ms'))
display(plot_trajectory_3d(
    dataset, x='trial_time_s', y='current_angle', z='dt_ms', color='run_id'
))

## 5. Select raw data for a custom check

Filters compose across runs, levels, and trial IDs. No data is copied until the final filtered DataFrame is returned.

In [ ]:
example_run = dataset.run_summary.iloc[0].run_id
selected = filter_frames(dataset.frames, run_ids=[example_run], levels=[0])
display(selected[['trial_uid', 'frame_number', 'present_elapsed_secs', 'dt_ms',
                  'late_frame', 'cmd_rotate_left', 'cmd_rotate_right',
                  'cmd_check', 'check_input_event_elapsed_secs',
                  'current_angle']].head(20))

## Interpretation guide

- **Counter gap:** `frame_number` or `render_frame_number` jumps by more than one. This means rows are absent from the saved log.
- **Timing stall:** `present_elapsed_secs` has a long delta even when counters remain consecutive. Frame counters count completed game/render loops, not every display-refresh opportunity, so a 100 ms frame still advances the counter by one.
- **Late frame:** by default, a positive timestamp interval above 25 ms (`1.5 × 1/60 s`). Change `AnalysisConfig` for another display rate or threshold.
- **Drift:** logged elapsed time minus `frame_index / expected_hz`. It is only meaningful relative to the selected expected refresh rate.
- **Input response:** a logged left/right command is compared with the following row's angle. A non-response is diagnostic evidence, not automatically proof of lost input. Ring-buffer catch-up recovers game state but cannot recover per-controller-tick `commands_sent`, so those recovered rows are saved with false command flags.
- **Presentation timestamp:** `present_elapsed_secs` is paired by render-frame ID and stamped immediately after wgpu `present()`. It is the closest portable software marker in this stack, but compositor/scanout delay remains and only the photodiode establishes physical photon onset.
- **Check timestamp:** new logs add nullable `check_input_event_elapsed_secs` beside `commands_sent`. It uses the controller's monotonic clock (mapped to Unix by `session_info.app_start_unix_ns`) and marks the human event consumed by that dispatch row. It is intentionally not a shared-memory state field.
- **Level `timing_health`:** current controllers compute deltas within each trial, use the pooled median for measured refresh, and populate late-interval, freeze, and drift fields. Older summaries may contain the former cross-boundary/hardcoded values; this notebook always recomputes from frame rows.
- **Mobile touch:** touch rotation is pulse-width-modulated, so false rows between true rotation rows can be intentional. Do not interpret them like gaps in a continuously held keyboard key.